# Full Coupled ODE

In [6]:
import constants as c
import numpy as np
import math
import matplotlib.pyplot as plt
import time
import numpy as np
import pandas as pd
import csv
from scipy.stats import maxwell
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp
from concurrent.futures import ProcessPoolExecutor, as_completed

vk = c.vk
vrf = c.vrf
z_offset = c.z0
z_min_valid = -c.ion_height
z_margin = 1.0e-6   # optional safety margin away from electrode plane
xy1k = c.xy1k
xy2k = c.xy2k
pi = math.pi
um = c.um
cm = c.cm
mm = c.mm
K = c.K
e = c.e
alpha = 1/137.06
d_chain = 3.7066438742e-06  # 3.7 µm separation between ions
delta_ion_2 = 0#-4.9903032463e-14 # small displacement of ion 2
m_ion = c.m 
Z_ion = c.Z
freq_x = 719430.7131391969
freq_y = 3031200.0099101723
freq_z = 3002153.5607483205
omega_vec = np.asarray(2*pi*np.array([freq_x,freq_y,freq_z]), dtype=float) 
hbar = 1.0545718e-34 
mode_y = 2*pi*np.array([3031200.01,2944587.06,2818861.50])
mode_z = 2*pi*np.array([3002153.56,2914677.59,2787603.39])

In [7]:
def run_simulation_many_dm_single_ion(
    x0_dms, v0_dms, x0_ion, v0_ion,
    t_span, rtol, atol, dt, m_dm, eps, run_rutherford
):
    """
    Run a single-ion / many-DM trajectory simulation.

    This version includes:
        - one trapped ion near the origin
        - many DM particles
        - ion-DM Coulomb forces
        - no DM-DM Coulomb forces

    Parameters
    ----------
    x0_dms : array-like, shape (n_dm, 3)
        Initial positions of the DM particles.

    v0_dms : array-like, shape (n_dm, 3)
        Initial velocities of the DM particles.

    x0_ion : array-like, shape (3,)
        Initial position of the single ion.

    v0_ion : array-like, shape (3,)
        Initial velocity of the single ion.

    t_span : tuple (t_start, t_end)
        Start and end time of the integration.

    dt : float
        Time increment used to build the evaluation grid.

    m_dm : float
        Mass of DM particles in kg.
    
    eps : float
        Charge of DM particles in units of elementary charge.

    run_rutherford : bool
        If True, set trap fields to zero for Rutherford scattering comparison.

    Returns
    -------
    t_eval : ndarray, shape (N,)
        Times at which the solution was evaluated.

    x_ion_sol : ndarray, shape (3, N)
        Ion position trajectory.

    v_ion_sol : ndarray, shape (3, N)
        Ion velocity trajectory.

    x_dms_sol : ndarray, shape (n_dm, 3, N)
        DM position trajectories.

    v_dms_sol : ndarray, shape (n_dm, 3, N)
        DM velocity trajectories.
    """

    x0_dms = np.asarray(x0_dms, dtype=float)
    v0_dms = np.asarray(v0_dms, dtype=float)
    x0_ion = np.asarray(x0_ion, dtype=float)
    v0_ion = np.asarray(v0_ion, dtype=float)

    if x0_dms.ndim != 2 or x0_dms.shape[1] != 3:
        raise ValueError("x0_dms must have shape (n_dm, 3)")

    if v0_dms.shape != x0_dms.shape:
        raise ValueError("v0_dms must have the same shape as x0_dms")

    if x0_ion.shape != (3,):
        raise ValueError("x0_ion must have shape (3,)")

    if v0_ion.shape != (3,):
        raise ValueError("v0_ion must have shape (3,)")

    n_dm = x0_dms.shape[0]

    # The single ion is trapped about the origin.
    r_ion_eq = np.zeros(3, dtype=float)

    def rhs_unified(t, U, r_ion_eq, use_harmonic_ion=True):
        """
        State vector layout:

        U = [
            x_ion(3),
            x_dms(3*n_dm),
            v_ion(3),
            v_dms(3*n_dm)
        ]
        """

        idx0 = 0
        idx1 = idx0 + 3
        idx2 = idx1 + 3 * n_dm
        idx3 = idx2 + 3
        idx4 = idx3 + 3 * n_dm

        x_ion = U[idx0:idx1]
        x_dms = U[idx1:idx2].reshape((n_dm, 3))

        v_ion = U[idx2:idx3]
        v_dms = U[idx3:idx4].reshape((n_dm, 3))

        dx_iondt = v_ion
        dx_dmsdt = v_dms

        dv_iondt = np.zeros(3)
        dv_dmsdt = np.zeros((n_dm, 3))

        # --------------------------------------------------
        # Trap force acting on ion
        # --------------------------------------------------
        if use_harmonic_ion:
            if run_rutherford:
                dv_iondt += np.zeros(3)
            else:
                dv_iondt += -omega_vec**2 * (x_ion - r_ion_eq)
        else:
            if not run_rutherford:
                x_ion_arr = x_ion[None, :]
                F_dc_ion = FDC(x_ion_arr, m=m_ion, Z=Z_ion)[0]
                F_rf_ion = FRF(x_ion_arr, m=m_ion, Z=Z_ion)[0]
                F_ion = F_dc_ion + F_rf_ion
                dv_iondt += F_ion / m_ion

        # --------------------------------------------------
        # Trap force acting on each DM particle
        # --------------------------------------------------
        if not run_rutherford:
            F_dc_dms = FDC(x_dms, m_dm, eps)
            F_rf_dms = FRF(x_dms, m_dm, eps)

            dv_dmsdt += (F_dc_dms + F_rf_dms) / m_dm

        # --------------------------------------------------
        # Coulomb interactions: ion <-> each DM particle
        #
        # No DM-DM Coulomb forces are included.
        # --------------------------------------------------
        delta = x_ion[None, :] - x_dms
        r = np.linalg.norm(delta, axis=1)

        forces = K * Z_ion * eps * e**2 * delta / r[:, None]**3

        # Force on ion due to DM k
        dv_iondt += forces.sum(axis=0) / m_ion

        # Equal and opposite force on DM k due to ion
        dv_dmsdt -= forces / m_dm

        return np.concatenate([
            dx_iondt,
            dx_dmsdt.flatten(),
            dv_iondt,
            dv_dmsdt.flatten()
        ])

    print("Number of DM particles:", n_dm)
    print("Initial ion position:", x0_ion)
    print("Initial ion velocity:", v0_ion)
    if(x0_dms.shape==(1,3)):
        print("Initial DM position:", x0_dms)
        print("Initial DM velocity:", v0_dms)

    # Initial state
    U0 = np.concatenate([
        x0_ion,
        x0_dms.flatten(),
        v0_ion,
        v0_dms.flatten()
    ]).astype(float)

    # Time grid
    t_eval = np.arange(t_span[0], t_span[1], dt)

    # Solve
    sol = solve_ivp(
        rhs_unified,
        t_span,
        U0,
        method="DOP853",
        t_eval=t_eval,
        dense_output=True,
        rtol=rtol,
        atol=atol,
        max_step=dt,
        args=(r_ion_eq,)
    )

    print("Success?", sol.success)
    print("Message:", sol.message)

    if sol.t.size > 0:
        print("Final time:", sol.t[-1])

    # Extract solution
    idx0 = 0
    idx1 = idx0 + 3
    idx2 = idx1 + 3 * n_dm
    idx3 = idx2 + 3
    idx4 = idx3 + 3 * n_dm

    x_ion_sol = sol.y[idx0:idx1]
    x_dms_flat_sol = sol.y[idx1:idx2]

    v_ion_sol = sol.y[idx2:idx3]
    v_dms_flat_sol = sol.y[idx3:idx4]

    # Convert flattened DM solution into shape (n_dm, 3, N)
    n_t = sol.y.shape[1]

    x_dms_sol = x_dms_flat_sol.reshape((n_dm, 3, n_t))
    v_dms_sol = v_dms_flat_sol.reshape((n_dm, 3, n_t))

    return t_eval, x_ion_sol, v_ion_sol, x_dms_sol, v_dms_sol

In [8]:
def analyze_simulation_results(x0_dms, v0_dms, x0_ion, v0_ion, t_span, dt, t_min, m_dm, eps, fit_curve, rutherford, show_plots,
                                rtol, atol):
    # default rtol=1e-13, atol=1e-16
    if(rutherford):
        t00, xion00, vion00, xdms00, vdms00 = run_simulation_many_dm_single_ion(x0_dms, v0_dms, x0_ion, v0_ion, t_span, rtol,atol,dt, m_dm, eps, run_rutherford=True)
    else:
        t00, xion00, vion00, xdms00, vdms00 = run_simulation_many_dm_single_ion(x0_dms, v0_dms, x0_ion, v0_ion, t_span, rtol,atol,dt, m_dm, eps, run_rutherford=False)
    t0 = t00/um
    xion0 = xion00/um
    vion0 = vion00
    xdms0 = xdms00/um
    vdms0 = vdms00
    x0_ion1, y0_ion1, z0_ion1 = xion0[0], xion0[1], xion0[2] # motion of center ion
    d_dms = np.linalg.norm(xdms0, axis=1) # for DM

    ## Plots 
    if(show_plots):

        # -------------------------
        # Ion coordinates vs time
        # -------------------------
        plt.figure(figsize=(18, 5))

        plt.subplot(1, 3, 1)
        plt.plot(t0, x0_ion1, label="x ion")
        plt.xlabel("t (us)")
        plt.ylabel("x (um)")
        plt.title("Ion x-position vs time")
        plt.legend()

        plt.subplot(1, 3, 2)
        plt.plot(t0, y0_ion1, label="y ion")
        plt.xlabel("t (us)")
        plt.ylabel("y (um)")
        plt.title("Ion y-position vs time")
        plt.legend()

        plt.subplot(1, 3, 3)
        plt.plot(t0, z0_ion1, label="z ion")
        plt.xlabel("t (us)")
        plt.ylabel("z (um)")
        plt.title("Ion z-position vs time")
        plt.legend()

        plt.tight_layout()
        plt.show()

        # print for single DM
        if(np.array(x0_dms).shape==(1,3)):
            fig = plt.figure(figsize=(18,5))
            x0,y0,z0 = xdms0[0][0],xdms0[0][1],xdms0[0][2]

            ax = fig.add_subplot(projection='3d')
            ax.set_xlabel('x (um)')
            ax.set_ylabel('y (um)')
            ax.set_zlabel('z (um)')
            ax.plot3D(x0,y0,z0,'red')
            ax.scatter(0, 0, 0, color='blue', s=50) 
            plt.title("DM Trajectory")
            plt.legend()

            plt.figure(figsize=(18,5))
            plt.plot(t0,d_dms[0])
            plt.xlabel("t (us)")
            plt.ylabel("distance (um)")
            plt.title("DM distance vs time")

            plt.tight_layout()
            plt.show()

    return t00, xion00, vion00, xdms0, vdms0, d_dms

In [9]:
def make_trap_anisotropic_trajectory(theta, alpha, psi, b, speed, R_start):
    """
    One-DM initial condition for an anisotropic trap.

    Trap convention:
        x = axial direction
        y,z = radial directions, with omega_y != omega_z

    Parameters
    ----------
    theta : float
        Polar angle from the axial x-axis.

    alpha : float
        Azimuthal angle around the x-axis.
        alpha = 0 means radial projection along y.
        alpha = pi/2 means radial projection along z.

    psi : float
        Impact-parameter angle in the plane perpendicular to u_hat.

    b : float
        Impact parameter magnitude.

    speed : float
        DM speed.

    R_start : float
        Starting distance upstream along -u_hat.

    Returns
    -------
    r0_dm : ndarray, shape (3,)
        Initial DM position.

    v0_dm : ndarray, shape (3,)
        Initial DM velocity.

    b_vec : ndarray, shape (3,)
        Impact vector.

    u_hat : ndarray, shape (3,)
        Incoming velocity direction.
    """
    u_hat = np.array([
        np.cos(theta),
        np.sin(theta) * np.cos(alpha),
        np.sin(theta) * np.sin(alpha)
    ], dtype=float)

    e_theta = np.array([
        -np.sin(theta),
        np.cos(theta) * np.cos(alpha),
        np.cos(theta) * np.sin(alpha)
    ], dtype=float)

    e_alpha = np.array([
        0.0,
        -np.sin(alpha),
        np.cos(alpha)
    ], dtype=float)

    u_hat = u_hat / np.linalg.norm(u_hat)
    e_theta = e_theta / np.linalg.norm(e_theta)
    e_alpha = e_alpha / np.linalg.norm(e_alpha)

    b_vec = b * (np.cos(psi) * e_theta + np.sin(psi) * e_alpha)

    r0_dm = b_vec - R_start * u_hat
    v0_dm = speed * u_hat

    return r0_dm, v0_dm, b_vec, u_hat, e_theta, e_alpha

## Run Simulation

In [10]:
def single_simulation(m_dm,eps,t_min,t_max,rtol,atol,dt,theta,alpha,psi,R,b,plot,speed=None,target_E=1e-27,seed=1,n=1,T=300):
    t_span = (t_min, t_max)
    if speed is None:
        speed_sample = sample_maxwell_boltzmann(
            n=n,
            T=T,
            m=m_dm,
            plot=False,
            seed=seed
        )[1]
        speed = float(np.ravel(speed_sample)[0])
    else:
        speed = float(speed)

    # Initial conditions
    x0_ion = np.array([0.0, 0.0, 0.0])
    v0_ion = np.array([0.0, 0.0, 0.0])

    x0_dms, v0_dms, b_vec, u_hat, e_theta, e_alpha = make_trap_anisotropic_trajectory(theta, alpha, psi, b, speed, R)
    x0_dms = [x0_dms]
    v0_dms = [v0_dms]

    # Run simulation
    start_time = time.perf_counter()

    print(
        f"--------Simulation Parameters--------\n"
        f"m_dm={m_dm:.2e}, eps={eps:.2e}, \n"
        f"rtol={rtol:.0e}, atol={atol:.0e}, dt={dt:.1e}, \n"
        f"theta={theta: .2e}, alpha={alpha: .2e}, psi={psi: .2e}, \n"
        f"R={R/um: .2e} um, b={b/um: .2e} um, \n"
        f"--------Simulation Progress--------"
    )

    t_eval, x_ion, v_ion, x_dms, v_dms, d_dm = analyze_simulation_results(
        x0_dms,
        v0_dms,
        x0_ion,
        v0_ion,
        t_span=t_span,
        dt=dt,
        t_min=t_min,
        m_dm=m_dm,
        eps=eps,
        fit_curve=False,
        rutherford=False,
        show_plots=plot,
        rtol=rtol,
        atol=atol
    )

    runtime = time.perf_counter() - start_time

    # Find closest approach
    ## print for single ion, single DM
    if(np.array(x0_dms).shape==(1,3)):
        r_threshold = 100
        interaction_time = (d_dm<r_threshold).sum()*dt
        d_min = np.min(d_dm)
    
    # Compute energy
    E_modes = (
        0.5 * m_ion * v_ion**2
        + 0.5 * m_ion * (omega_vec[:, None] * x_ion)**2
    )
    print(E_modes[:,-1])

    E_ion = np.sum(E_modes, axis=0)
    E_final = E_ion[-1]

    # Print simulation results
    print(
        f"--------Simulation Results--------\n"
        f"E={E_final:.6e} J, "
        f"above={E_final >= target_E}, \n"
        f"Ending position of DM: {x_dms[0, :, -1]} um, \n"
        f"Ending velocity of DM: {v_dms[0, :, -1]} um/us, \n"
        f"Interaction time for threshold {r_threshold} um: {interaction_time} us, \n"
        f"Closest approach of DM to origin: {d_min} um, \n"
        f"runtime={runtime:.2f} s\n"
    )

    return E_modes, E_final, d_min, runtime

# Optimized Staged ODE for Far-Start DM

In [12]:
#!/usr/bin/env python3
"""Three-stage single-DM / single-ion trajectory simulation.

This module combines the handoff logic used by the Test 1 and Test 2 scripts:

1. Test 1 outer-tail straight-line diagnostic
   The nominal incoming line is sampled between ``R_full`` and ``R_max``.
   The smallest safe outer-tail handoff point is called ``R_far``.  The DM
   speed at that point is corrected by conservation of energy in the static
   trap potential.

2. Test 2 DM-only trap propagation
   The DM is propagated from the Test 1 ``R_far`` state until it enters the
   sphere ``R_switch``.  The ion is not included in this stage.

3. Full ion-DM coupled propagation
   At the incoming ``R_switch`` crossing, the ion is introduced and the full
   ion-trap + DM-trap + ion-DM Coulomb system is integrated until the pair
   escapes the local interaction region or the integration times out.

The main public entry point is ``single_simulation``.  With ``plot=True``
the module also displays ion motion in the xy, yz and xz planes.  The
printed summary reports the three handoff radii, their reach times, and
the initial/final ion and DM phase-space states.

Required trap-model objects
---------------------------
The easiest notebook workflow is to run the trap-model cells first and then:

    %run -i modified_staged_single_simulation.py

The following names are then resolved from the notebook namespace:

    potential_energy, force, FDC, FRF,
    m_ion, Z_ion, omega_vec, K, e

Alternatively, construct a ``SimulationEnvironment`` and pass it through the
``environment=`` argument.

Important convention
--------------------
The incoming direction is

    u = [cos(theta), sin(theta) cos(alpha), sin(theta) sin(alpha)]

and the nominal incoming line is

    r(s) = b_vec - s u,

with velocity directed along ``+u``.  Consequently, the initial radial
velocity is negative for an incoming trajectory.
"""

from __future__ import annotations

from dataclasses import dataclass
import math
import time
from typing import Any, Callable

import numpy as np
from scipy.integrate import solve_ivp


Array = np.ndarray


# =============================================================================
# CONFIGURATION DATA CLASSES
# =============================================================================


@dataclass
class SimulationEnvironment:
    """Trap-model functions and constants required by the staged solver."""

    potential_energy: Callable[..., Any]
    force: Callable[..., Any] | None
    FDC: Callable[..., Any]
    FRF: Callable[..., Any]
    m_ion: float
    Z_ion: float
    omega_vec: Array
    K: float
    e: float
    ion_height_m: float | None = None

    @classmethod
    def from_namespace(cls, namespace: dict[str, Any]) -> "SimulationEnvironment":
        required = (
            "potential_energy",
            "FDC",
            "FRF",
            "m_ion",
            "Z_ion",
            "omega_vec",
            "K",
            "e",
        )
        missing = [name for name in required if name not in namespace]
        if missing:
            raise RuntimeError(
                "Missing trap-model names: " + ", ".join(missing)
            )

        ion_height_m = None
        c_obj = namespace.get("c")
        if c_obj is not None and hasattr(c_obj, "ion_height"):
            ion_height_m = float(c_obj.ion_height)

        return cls(
            potential_energy=namespace["potential_energy"],
            force=namespace.get("force"),
            FDC=namespace["FDC"],
            FRF=namespace["FRF"],
            m_ion=float(namespace["m_ion"]),
            Z_ion=float(namespace["Z_ion"]),
            omega_vec=np.asarray(namespace["omega_vec"], dtype=float).reshape(3),
            K=float(namespace["K"]),
            e=float(namespace["e"]),
            ion_height_m=ion_height_m,
        )


@dataclass
class OuterTailSettings:
    R_max_m: float = 20.0e-3
    n_path: int = 800
    energy_tol: float = 1.0e-2
    angle_tol: float = 1.0e-3
    displacement_tol_m: float = 1.0e-6
    U_reference_J: float = 0.0
    z_margin_m: float = 1.0e-6
    allow_R_far_lower_bound: bool = False


@dataclass
class DMOnlySettings:
    rtol: float = 1.0e-6
    atol: float = 1.0e-9
    max_step_s: float = 5.0e-7
    time_factor: float = 3.0
    minimum_after_arrival_s: float = 20.0e-6
    escape_factor_from_R_far: float = 1.05
    local_escape_factor_from_R_switch: float = 1.5


@dataclass
class FullSettings:
    rtol: float = 1.0e-6
    atol: float = 1.0e-9
    sample_dt_s: float = 5.0e-9
    max_step_s: float = 5.0e-9
    time_factor: float = 8.0
    minimum_time_s: float = 20.0e-6
    escape_factor_from_R_switch: float = 1.5
    energy_average_window_s: float = 2.0e-6
    use_harmonic_ion: bool = True
    coulomb_softening_m: float = 0.0


# =============================================================================
# GENERAL HELPERS
# =============================================================================


def _resolve_environment(
    environment: SimulationEnvironment | None,
) -> SimulationEnvironment:
    if environment is not None:
        return environment
    return SimulationEnvironment.from_namespace(globals())


def _as_vector(value: Any, name: str) -> Array:
    vector = np.asarray(value, dtype=float).reshape(-1)
    if vector.shape != (3,) or not np.all(np.isfinite(vector)):
        raise ValueError(f"{name} must be a finite length-3 vector")
    return vector


def radial_velocity(position: Array, velocity: Array) -> float:
    position = _as_vector(position, "position")
    velocity = _as_vector(velocity, "velocity")
    radius = float(np.linalg.norm(position))
    if radius <= 0.0:
        return 0.0
    return float(np.dot(position, velocity) / radius)


def _append_event_state(
    solution: Any,
    event_index: int,
) -> tuple[Array, Array]:
    times = np.asarray(solution.t, dtype=float)
    states = np.asarray(solution.y, dtype=float)
    if (
        solution.t_events is None
        or len(solution.t_events) <= event_index
        or len(solution.t_events[event_index]) == 0
    ):
        return times, states

    event_time = float(solution.t_events[event_index][0])
    event_state = np.asarray(solution.y_events[event_index][0], dtype=float)
    if times.size == 0 or not np.isclose(times[-1], event_time):
        times = np.append(times, event_time)
        states = np.column_stack((states, event_state))
    return times, states


def _sampled_min_radius(states: Array, position_slice: slice) -> float:
    if states.size == 0:
        return float("nan")
    radius = np.linalg.norm(np.asarray(states[position_slice], dtype=float), axis=0)
    finite = radius[np.isfinite(radius)]
    return float(np.min(finite)) if finite.size else float("nan")


def _time_inside(times: Array, mask: Array) -> float:
    times = np.asarray(times, dtype=float)
    mask = np.asarray(mask, dtype=bool)
    if times.size < 2:
        return 0.0
    dt = np.diff(times)
    active = mask[:-1] | mask[1:]
    return float(np.sum(dt[active]))


def _format_vector(vector: Any, *, scale: float = 1.0, precision: int = 6) -> str:
    values = np.asarray(vector, dtype=float).reshape(3) * float(scale)
    return np.array2string(
        values,
        precision=precision,
        suppress_small=False,
        separator=", ",
    )


def _first_event_time(solution: Any, event_index: int) -> float:
    if (
        solution.t_events is None
        or len(solution.t_events) <= event_index
        or len(solution.t_events[event_index]) == 0
    ):
        return float("nan")
    return float(solution.t_events[event_index][0])


# =============================================================================
# TRAJECTORY GEOMETRY
# =============================================================================


def direction_basis(
    theta: float,
    alpha: float,
    psi: float,
) -> tuple[Array, Array, Array, Array]:
    """Return incoming direction and the perpendicular impact basis."""

    u_hat = np.array(
        [
            math.cos(theta),
            math.sin(theta) * math.cos(alpha),
            math.sin(theta) * math.sin(alpha),
        ],
        dtype=float,
    )
    e_theta = np.array(
        [
            -math.sin(theta),
            math.cos(theta) * math.cos(alpha),
            math.cos(theta) * math.sin(alpha),
        ],
        dtype=float,
    )
    e_alpha = np.array(
        [0.0, -math.sin(alpha), math.cos(alpha)],
        dtype=float,
    )

    u_hat /= np.linalg.norm(u_hat)
    e_theta /= np.linalg.norm(e_theta)
    e_alpha /= np.linalg.norm(e_alpha)

    b_hat = math.cos(psi) * e_theta + math.sin(psi) * e_alpha
    b_hat /= np.linalg.norm(b_hat)
    return u_hat, b_hat, e_theta, e_alpha


def build_straight_path(
    *,
    theta: float,
    alpha: float,
    psi: float,
    b_m: float,
    R_inner_m: float,
    R_outer_m: float,
    n_path: int,
) -> dict[str, Array | float]:
    """Construct the Test 1 nominal incoming line.

    ``s`` increases outward.  The particle moves inward along ``+u_hat``.
    For ``b < R_inner``, the innermost point is the line's intersection with
    the ``R_inner`` sphere.  If the line misses that sphere, the diagnostic is
    extended close to nominal closest approach, matching the Test 1 logic.
    """

    b_m = float(b_m)
    R_inner_m = float(R_inner_m)
    R_outer_m = float(R_outer_m)
    n_path = int(n_path)

    if b_m < 0.0:
        raise ValueError("b_m must be nonnegative")
    if R_inner_m <= 0.0:
        raise ValueError("R_inner_m must be positive")
    if R_outer_m <= R_inner_m:
        raise ValueError("R_outer_m must exceed R_inner_m")
    if b_m >= R_outer_m:
        raise ValueError("b_m must be smaller than R_outer_m")
    if n_path < 3:
        raise ValueError("n_path must be at least 3")

    u_hat, b_hat, e_theta, e_alpha = direction_basis(theta, alpha, psi)
    b_vec = b_m * b_hat

    s_outer = math.sqrt(max(R_outer_m**2 - b_m**2, 0.0))
    if b_m < R_inner_m:
        s_inner = math.sqrt(max(R_inner_m**2 - b_m**2, 0.0))
    else:
        s_inner = max(1.0e-9, 0.02 * R_inner_m)

    if not (s_outer > s_inner > 0.0):
        raise ValueError(
            "Invalid straight-path coordinate interval: "
            f"s_inner={s_inner:.6e}, s_outer={s_outer:.6e}, b={b_m:.6e}"
        )

    s_grid = np.geomspace(s_inner, s_outer, n_path)
    xyz = b_vec[None, :] - s_grid[:, None] * u_hat[None, :]
    return {
        "s_grid_m": s_grid,
        "xyz_m": xyz,
        "u_hat": u_hat,
        "b_hat": b_hat,
        "b_vec_m": b_vec,
        "e_theta": e_theta,
        "e_alpha": e_alpha,
    }


# =============================================================================
# TRAP MODEL ADAPTERS
# =============================================================================


def _valid_xyz_mask(
    xyz: Array,
    *,
    invalid_z_floor_m: float | None,
) -> Array:
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    finite = np.all(np.isfinite(xyz), axis=1)
    if invalid_z_floor_m is None:
        return finite
    return finite & (xyz[:, 2] > float(invalid_z_floor_m))


def _evaluate_potential(
    points: Array,
    *,
    m_dm: float,
    eps: float,
    environment: SimulationEnvironment,
) -> Array:
    points = np.asarray(points, dtype=float).reshape(-1, 3)
    try:
        values = np.asarray(
            environment.potential_energy(
                points,
                m_dm,
                eps,
                component="trap",
            ),
            dtype=float,
        ).reshape(-1)
        if values.shape[0] != points.shape[0]:
            raise ValueError("Unexpected vectorized potential shape")
        return values
    except Exception:
        return np.array(
            [
                float(
                    np.asarray(
                        environment.potential_energy(
                            point,
                            m_dm,
                            eps,
                            component="trap",
                        )
                    ).reshape(-1)[0]
                )
                for point in points
            ],
            dtype=float,
        )


def _trap_force_from_FDC_FRF(
    points: Array,
    *,
    mass: float,
    charge: float,
    environment: SimulationEnvironment,
) -> Array:
    points = np.asarray(points, dtype=float).reshape(-1, 3)
    return (
        np.asarray(environment.FDC(points, mass, charge), dtype=float).reshape(-1, 3)
        + np.asarray(environment.FRF(points, mass, charge), dtype=float).reshape(-1, 3)
    )


def _evaluate_force(
    points: Array,
    *,
    mass: float,
    charge: float,
    environment: SimulationEnvironment,
) -> Array:
    points = np.asarray(points, dtype=float).reshape(-1, 3)

    if environment.force is not None:
        try:
            values = np.asarray(
                environment.force(
                    points,
                    mass,
                    charge,
                    component="trap",
                ),
                dtype=float,
            ).reshape(-1, 3)
            if values.shape[0] != points.shape[0]:
                raise ValueError("Unexpected vectorized force shape")
            return values
        except Exception:
            try:
                return np.vstack(
                    [
                        np.asarray(
                            environment.force(
                                point,
                                mass,
                                charge,
                                component="trap",
                            ),
                            dtype=float,
                        ).reshape(3)
                        for point in points
                    ]
                )
            except Exception:
                pass

    return _trap_force_from_FDC_FRF(
        points,
        mass=mass,
        charge=charge,
        environment=environment,
    )


def evaluate_straight_trap_path(
    xyz_m: Array,
    *,
    m_dm: float,
    eps: float,
    environment: SimulationEnvironment,
    invalid_z_floor_m: float | None,
) -> tuple[Array, Array, Array]:
    """Evaluate trap-only potential and force along the nominal line."""

    xyz_m = np.asarray(xyz_m, dtype=float).reshape(-1, 3)
    n_points = xyz_m.shape[0]
    potential = np.full(n_points, np.nan, dtype=float)
    force = np.full((n_points, 3), np.nan, dtype=float)

    geometry_valid = _valid_xyz_mask(
        xyz_m,
        invalid_z_floor_m=invalid_z_floor_m,
    )
    indices = np.flatnonzero(geometry_valid)
    if indices.size == 0:
        return potential, force, np.zeros(n_points, dtype=bool)

    valid_points = xyz_m[indices]
    potential[indices] = _evaluate_potential(
        valid_points,
        m_dm=m_dm,
        eps=eps,
        environment=environment,
    )
    force[indices] = _evaluate_force(
        valid_points,
        mass=m_dm,
        charge=eps,
        environment=environment,
    )

    valid = (
        geometry_valid
        & np.isfinite(potential)
        & np.all(np.isfinite(force), axis=1)
    )
    return potential, force, valid


# =============================================================================
# TEST 1 OUTER-TAIL R_FAR DIAGNOSTIC
# =============================================================================


def _reverse_cumulative_sum(segment_values: Array) -> Array:
    values = np.asarray(segment_values)
    if values.ndim == 1:
        output = np.zeros(values.shape[0] + 1, dtype=float)
        output[:-1] = np.cumsum(values[::-1])[::-1]
        return output

    output = np.zeros((values.shape[0] + 1,) + values.shape[1:], dtype=float)
    output[:-1] = np.cumsum(values[::-1], axis=0)[::-1]
    return output


def analyze_R_far(
    *,
    s_grid_m: Array,
    xyz_m: Array,
    u_hat: Array,
    potential_J: Array,
    force_N: Array,
    valid: Array,
    m_dm: float,
    v_inf_m_s: float,
    R_full_m: float,
    R_max_m: float,
    energy_tol: float,
    angle_tol: float,
    displacement_tol_m: float,
    U_reference_J: float = 0.0,
) -> dict[str, Any]:
    """Find the smallest Test 1 handoff point with a negligible outer tail."""

    s_grid = np.asarray(s_grid_m, dtype=float)
    xyz = np.asarray(xyz_m, dtype=float)
    u = _as_vector(u_hat, "u_hat")
    potential = np.asarray(potential_J, dtype=float)
    force = np.asarray(force_N, dtype=float)
    valid = np.asarray(valid, dtype=bool)

    kinetic_inf = 0.5 * float(m_dm) * float(v_inf_m_s) ** 2
    if not np.isfinite(kinetic_inf) or kinetic_inf <= 0.0:
        return {
            "outer_status": "invalid_speed",
            "R_far_m": np.nan,
            "R_far_actual_m": np.nan,
            "R_far_is_lower_bound": False,
            "v_far_m_s": np.nan,
            "limiting_criterion": "not_applicable",
        }

    if xyz.shape[0] == 0 or not np.any(valid):
        return {
            "outer_status": "no_valid_points",
            "R_far_m": np.nan,
            "R_far_actual_m": np.nan,
            "R_far_is_lower_bound": False,
            "v_far_m_s": np.nan,
            "limiting_criterion": "not_applicable",
        }
    if not valid[-1]:
        return {
            "outer_status": "invalid_far_start_below_electrode",
            "R_far_m": np.nan,
            "R_far_actual_m": np.nan,
            "R_far_is_lower_bound": False,
            "v_far_m_s": np.nan,
            "limiting_criterion": "not_applicable",
            "xyz_far_m": xyz[-1].copy(),
        }

    kinetic_local = kinetic_inf + float(U_reference_J) - potential
    point_ok = valid & np.isfinite(kinetic_local) & (kinetic_local > 0.0)

    local_speed_sq = np.full(s_grid.shape[0], np.nan, dtype=float)
    local_speed_sq[point_ok] = 2.0 * kinetic_local[point_ok] / float(m_dm)

    force_parallel_scalar = force @ u
    force_parallel = force_parallel_scalar[:, None] * u[None, :]
    force_perpendicular = force - force_parallel

    q_vector = np.zeros_like(force_perpendicular)
    q_vector[point_ok] = force_perpendicular[point_ok] / (
        float(m_dm) * local_speed_sq[point_ok, None]
    )
    q_abs = np.linalg.norm(q_vector, axis=1)

    ds = np.diff(s_grid)
    s_mid = 0.5 * (s_grid[:-1] + s_grid[1:])
    segment_ok = point_ok[:-1] & point_ok[1:]

    q_mid_vector = 0.5 * (q_vector[:-1] + q_vector[1:])
    q_mid_abs = 0.5 * (q_abs[:-1] + q_abs[1:])
    q_mid_vector[~segment_ok] = 0.0
    q_mid_abs[~segment_ok] = 0.0

    theta_vector_tail = _reverse_cumulative_sum(q_mid_vector * ds[:, None])
    theta_signed_tail = np.linalg.norm(theta_vector_tail, axis=1)
    theta_abs_tail = _reverse_cumulative_sum(q_mid_abs * ds)

    moment_vector_tail = _reverse_cumulative_sum(
        s_mid[:, None] * q_mid_vector * ds[:, None]
    )
    displacement_vector_tail = (
        moment_vector_tail - s_grid[:, None] * theta_vector_tail
    )
    displacement_signed_tail = np.linalg.norm(displacement_vector_tail, axis=1)

    moment_abs_tail = _reverse_cumulative_sum(s_mid * q_mid_abs * ds)
    displacement_abs_tail = np.maximum(
        moment_abs_tail - s_grid * theta_abs_tail,
        0.0,
    )

    valid_tail = np.logical_and.accumulate(valid[::-1])[::-1]
    kinetic_for_tail = np.where(valid, kinetic_local, -np.inf)
    kinetic_tail_min = np.minimum.accumulate(kinetic_for_tail[::-1])[::-1]

    energy_ratio_point = np.where(
        valid,
        np.abs(potential - float(U_reference_J)) / kinetic_inf,
        np.inf,
    )
    energy_ratio_tail_max = np.maximum.accumulate(
        energy_ratio_point[::-1]
    )[::-1]

    safe = (
        valid_tail
        & (kinetic_tail_min > 0.0)
        & (energy_ratio_tail_max <= float(energy_tol))
        & (theta_abs_tail <= float(angle_tol))
        & (displacement_abs_tail <= float(displacement_tol_m))
    )
    safe_indices = np.flatnonzero(safe)

    nonpositive = np.flatnonzero(
        valid & np.isfinite(kinetic_local) & (kinetic_local <= 0.0)
    )
    turning_radius_m = (
        float(np.max(s_grid[nonpositive])) if nonpositive.size else np.nan
    )

    if safe_indices.size == 0:
        limiting = (
            "R_max_nonpositive_K" if not point_ok[-1] else "R_max_energy"
        )
        v_at_max = (
            math.sqrt(2.0 * kinetic_local[-1] / float(m_dm))
            if point_ok[-1]
            else np.nan
        )
        return {
            "outer_status": "needs_larger_R_max",
            "R_far_m": float(R_max_m),
            "R_far_actual_m": float(np.linalg.norm(xyz[-1])),
            "R_far_is_lower_bound": True,
            "v_far_m_s": v_at_max,
            "U_far_J": float(potential[-1]),
            "K_inf_J": kinetic_inf,
            "energy_tail_max": (
                float(energy_ratio_tail_max[-1])
                if np.isfinite(energy_ratio_tail_max[-1])
                else np.nan
            ),
            "theta_signed_tail": 0.0,
            "theta_abs_tail": 0.0,
            "displacement_signed_tail_m": 0.0,
            "displacement_abs_tail_m": 0.0,
            "limiting_criterion": limiting,
            "turning_radius_m": turning_radius_m,
            "xyz_far_m": xyz[-1].copy(),
            "velocity_far_m_s": (
                v_at_max * u if np.isfinite(v_at_max) else np.full(3, np.nan)
            ),
            "safe_mask": safe,
            "energy_ratio_tail_max_array": energy_ratio_tail_max,
            "theta_abs_tail_array": theta_abs_tail,
            "displacement_abs_tail_array_m": displacement_abs_tail,
        }

    index = int(safe_indices[0])
    K_far = float(kinetic_local[index])
    if K_far <= 0.0:
        return {
            "outer_status": "nonpositive_K_at_R_far",
            "R_far_m": float(s_grid[index]),
            "R_far_actual_m": float(np.linalg.norm(xyz[index])),
            "R_far_is_lower_bound": False,
            "v_far_m_s": np.nan,
            "limiting_criterion": "not_applicable",
            "turning_radius_m": turning_radius_m,
        }

    v_far = math.sqrt(2.0 * K_far / float(m_dm))
    normalized_metrics = {
        "energy": float(energy_ratio_tail_max[index]) / float(energy_tol),
        "angle": float(theta_abs_tail[index]) / float(angle_tol),
        "displacement": (
            float(displacement_abs_tail[index]) / float(displacement_tol_m)
        ),
    }
    limiting = max(normalized_metrics, key=normalized_metrics.get)
    outer_status = (
        "straight_safe_to_full_sphere" if index == 0 else "straight_then_dm_only"
    )

    return {
        "outer_status": outer_status,
        "R_far_m": float(s_grid[index]),
        "R_far_actual_m": float(np.linalg.norm(xyz[index])),
        "R_far_is_lower_bound": False,
        "v_far_m_s": v_far,
        "U_far_J": float(potential[index]),
        "K_inf_J": kinetic_inf,
        "energy_tail_max": float(energy_ratio_tail_max[index]),
        "theta_signed_tail": float(theta_signed_tail[index]),
        "theta_abs_tail": float(theta_abs_tail[index]),
        "displacement_signed_tail_m": float(displacement_signed_tail[index]),
        "displacement_abs_tail_m": float(displacement_abs_tail[index]),
        "limiting_criterion": limiting,
        "turning_radius_m": turning_radius_m,
        "xyz_far_m": xyz[index].copy(),
        "velocity_far_m_s": v_far * u,
        "safe_index": index,
        "safe_mask": safe,
        "energy_ratio_tail_max_array": energy_ratio_tail_max,
        "theta_abs_tail_array": theta_abs_tail,
        "displacement_abs_tail_array_m": displacement_abs_tail,
    }


def run_straight_outer_tail_test(
    *,
    theta: float,
    alpha: float,
    psi: float,
    b_m: float,
    v_inf_m_s: float,
    m_dm: float,
    eps: float,
    R_full_m: float,
    settings: OuterTailSettings,
    environment: SimulationEnvironment,
    invalid_z_floor_m: float | None,
) -> dict[str, Any]:
    """Run the complete Test 1 straight-line outer-tail stage."""

    geometry = build_straight_path(
        theta=theta,
        alpha=alpha,
        psi=psi,
        b_m=b_m,
        R_inner_m=R_full_m,
        R_outer_m=settings.R_max_m,
        n_path=settings.n_path,
    )
    potential, force, valid = evaluate_straight_trap_path(
        geometry["xyz_m"],
        m_dm=m_dm,
        eps=eps,
        environment=environment,
        invalid_z_floor_m=invalid_z_floor_m,
    )
    outer = analyze_R_far(
        s_grid_m=geometry["s_grid_m"],
        xyz_m=geometry["xyz_m"],
        u_hat=geometry["u_hat"],
        potential_J=potential,
        force_N=force,
        valid=valid,
        m_dm=m_dm,
        v_inf_m_s=v_inf_m_s,
        R_full_m=R_full_m,
        R_max_m=settings.R_max_m,
        energy_tol=settings.energy_tol,
        angle_tol=settings.angle_tol,
        displacement_tol_m=settings.displacement_tol_m,
        U_reference_J=settings.U_reference_J,
    )
    outer.update(
        {
            "s_grid_m": geometry["s_grid_m"],
            "xyz_path_m": geometry["xyz_m"],
            "u_hat": geometry["u_hat"],
            "b_hat": geometry["b_hat"],
            "b_vec_m": geometry["b_vec_m"],
            "potential_path_J": potential,
            "force_path_N": force,
            "valid_path": valid,
        }
    )
    return outer


# =============================================================================
# TEST 2 DM-ONLY PROPAGATION TO R_SWITCH
# =============================================================================


def _dm_only_rhs(
    *,
    m_dm: float,
    eps: float,
    environment: SimulationEnvironment,
) -> Callable[[float, Array], Array]:
    def rhs(_time: float, state: Array) -> Array:
        position = state[:3]
        velocity = state[3:]
        force = _trap_force_from_FDC_FRF(
            position[None, :],
            mass=m_dm,
            charge=eps,
            environment=environment,
        )[0]
        acceleration = force / float(m_dm)
        return np.concatenate((velocity, acceleration))

    return rhs


def _reconstruct_incoming_switch_state(
    x_inside_m: Array,
    v_inside_m_s: Array,
    *,
    m_dm: float,
    eps: float,
    R_switch_m: float,
    invalid_z_floor_m: float | None,
    settings: DMOnlySettings,
    environment: SimulationEnvironment,
) -> dict[str, Any]:
    """Integrate backward from an inside point to the incoming switch crossing."""

    speed = max(float(np.linalg.norm(v_inside_m_s)), 1.0e-12)
    time_scale = max(
        settings.time_factor * R_switch_m / speed,
        settings.minimum_after_arrival_s,
    )

    def switch_event(_time: float, state: Array) -> float:
        return float(np.linalg.norm(state[:3]) - R_switch_m)

    switch_event.terminal = True
    switch_event.direction = 0
    events: list[Callable] = [switch_event]

    if invalid_z_floor_m is not None:
        def invalid_event(_time: float, state: Array) -> float:
            return float(state[2] - invalid_z_floor_m)

        invalid_event.terminal = True
        invalid_event.direction = 0
        events.append(invalid_event)

    solution = solve_ivp(
        _dm_only_rhs(m_dm=m_dm, eps=eps, environment=environment),
        (0.0, -time_scale),
        np.concatenate((x_inside_m, v_inside_m_s)),
        method="DOP853",
        rtol=settings.rtol,
        atol=settings.atol,
        max_step=settings.max_step_s,
        events=events,
    )
    entered = len(solution.t_events[0]) > 0
    switch_time_relative = _first_event_time(solution, 0)
    if not entered:
        return {
            "success": False,
            "status": "dm_only_inside_requires_rebuild",
            "message": str(solution.message),
            "x_switch_m": np.full(3, np.nan),
            "v_switch_m_s": np.full(3, np.nan),
            "switch_time_relative_s": float("nan"),
            "solution": solution,
        }

    state = np.asarray(solution.y_events[0][0], dtype=float)
    return {
        "success": True,
        "status": "dm_only_reconstructed_incoming_switch",
        "message": str(solution.message),
        "x_switch_m": state[:3],
        "v_switch_m_s": state[3:],
        "switch_time_relative_s": switch_time_relative,
        "solution": solution,
    }


def run_dm_trap_only_to_switch(
    x0_m: Array,
    v0_m_s: Array,
    *,
    m_dm: float,
    eps: float,
    R_switch_m: float,
    settings: DMOnlySettings,
    environment: SimulationEnvironment,
    invalid_z_floor_m: float | None,
    keep_trajectory: bool = True,
) -> dict[str, Any]:
    """Propagate the DM under trap forces alone to the incoming switch sphere."""

    started = time.perf_counter()
    x0 = _as_vector(x0_m, "x0_m")
    v0 = _as_vector(v0_m_s, "v0_m_s")
    R_switch_m = float(R_switch_m)
    r0 = float(np.linalg.norm(x0))
    speed0 = max(float(np.linalg.norm(v0)), 1.0e-12)
    initial_radial_velocity = radial_velocity(x0, v0)

    if r0 < R_switch_m * (1.0 - 1.0e-10):
        rebuilt = _reconstruct_incoming_switch_state(
            x0,
            v0,
            m_dm=m_dm,
            eps=eps,
            R_switch_m=R_switch_m,
            invalid_z_floor_m=invalid_z_floor_m,
            settings=settings,
            environment=environment,
        )
        solution = rebuilt.pop("solution")
        switch_time_relative = float(rebuilt.pop("switch_time_relative_s", np.nan))
        x_switch = np.asarray(rebuilt["x_switch_m"], dtype=float)
        v_switch = np.asarray(rebuilt["v_switch_m_s"], dtype=float)
        result = {
            "dm_only_success": bool(rebuilt["success"]),
            "dm_only_status": str(rebuilt["status"]),
            "dm_only_message": str(rebuilt["message"]),
            "entered_switch": bool(rebuilt["success"]),
            "dm_only_runtime_s": time.perf_counter() - started,
            "dm_only_t_final_s": float(solution.t[-1]) if solution.t.size else 0.0,
            "dm_only_t_reach_R_switch_s": switch_time_relative,
            "dm_only_t_arrival_estimate_s": r0 / speed0,
            "dm_only_t_max_s": abs(float(solution.t[-1])) if solution.t.size else np.nan,
            "dm_only_initial_radius_m": r0,
            "dm_only_initial_radial_velocity_m_s": initial_radial_velocity,
            "dm_only_final_radius_m": (
                float(np.linalg.norm(solution.y[:3, -1])) if solution.y.size else r0
            ),
            "dm_only_min_radius_m": _sampled_min_radius(solution.y, slice(0, 3)),
            "dm_only_final_radial_velocity_m_s": (
                radial_velocity(solution.y[:3, -1], solution.y[3:, -1])
                if solution.y.size
                else np.nan
            ),
            "x_dm_initial_m": x0.copy(),
            "v_dm_initial_m_s": v0.copy(),
            "x_dm_final_m": (
                solution.y[:3, -1].copy() if solution.y.size else x0.copy()
            ),
            "v_dm_final_m_s": (
                solution.y[3:, -1].copy() if solution.y.size else v0.copy()
            ),
            "x_switch_m": x_switch,
            "v_switch_m_s": v_switch,
        }
        if keep_trajectory:
            result["times_s"] = np.asarray(solution.t, dtype=float)
            result["states"] = np.asarray(solution.y, dtype=float)
        return result

    if np.isclose(r0, R_switch_m, rtol=1.0e-10, atol=1.0e-15):
        return {
            "dm_only_success": True,
            "dm_only_status": "dm_only_started_on_switch",
            "dm_only_message": "Initial state lies on R_switch",
            "entered_switch": True,
            "dm_only_runtime_s": time.perf_counter() - started,
            "dm_only_t_final_s": 0.0,
            "dm_only_t_reach_R_switch_s": 0.0,
            "dm_only_t_arrival_estimate_s": 0.0,
            "dm_only_t_max_s": 0.0,
            "dm_only_initial_radius_m": r0,
            "dm_only_initial_radial_velocity_m_s": initial_radial_velocity,
            "dm_only_final_radius_m": r0,
            "dm_only_min_radius_m": r0,
            "dm_only_final_radial_velocity_m_s": initial_radial_velocity,
            "x_dm_initial_m": x0.copy(),
            "v_dm_initial_m_s": v0.copy(),
            "x_dm_final_m": x0.copy(),
            "v_dm_final_m_s": v0.copy(),
            "x_switch_m": x0.copy(),
            "v_switch_m_s": v0.copy(),
            "times_s": np.array([0.0]),
            "states": np.concatenate((x0, v0))[:, None],
        }

    arrival_estimate = r0 / speed0
    t_max = max(
        settings.time_factor * arrival_estimate,
        arrival_estimate + settings.minimum_after_arrival_s,
    )
    escape_radius = max(
        settings.escape_factor_from_R_far * r0,
        settings.local_escape_factor_from_R_switch * R_switch_m,
    )

    def switch_event(_time: float, state: Array) -> float:
        return float(np.linalg.norm(state[:3]) - R_switch_m)

    switch_event.terminal = True
    switch_event.direction = -1

    def escape_event(_time: float, state: Array) -> float:
        radius = float(np.linalg.norm(state[:3]))
        if radial_velocity(state[:3], state[3:]) <= 0.0:
            return -abs(radius - escape_radius) - 1.0e-30
        return radius - escape_radius

    escape_event.terminal = True
    escape_event.direction = 1

    def closest_approach_event(_time: float, state: Array) -> float:
        return radial_velocity(state[:3], state[3:])

    closest_approach_event.terminal = False
    closest_approach_event.direction = 1
    events: list[Callable] = [switch_event, escape_event, closest_approach_event]

    if invalid_z_floor_m is not None:
        def invalid_event(_time: float, state: Array) -> float:
            return float(state[2] - invalid_z_floor_m)

        invalid_event.terminal = True
        invalid_event.direction = -1
        events.append(invalid_event)

    solution = solve_ivp(
        _dm_only_rhs(m_dm=m_dm, eps=eps, environment=environment),
        (0.0, t_max),
        np.concatenate((x0, v0)),
        method="DOP853",
        rtol=settings.rtol,
        atol=settings.atol,
        max_step=settings.max_step_s,
        events=events,
    )

    entered = len(solution.t_events[0]) > 0
    escaped = len(solution.t_events[1]) > 0
    closest_states = (
        np.asarray(solution.y_events[2], dtype=float)
        if len(solution.y_events) > 2 and len(solution.y_events[2])
        else np.empty((0, 6), dtype=float)
    )
    invalid_index = 3
    invalid = len(events) > invalid_index and len(solution.t_events[invalid_index]) > 0
    reconstructed_message = ""
    switch_time_s = _first_event_time(solution, 0)

    if entered:
        status = "dm_only_entered_switch"
        event_state = np.asarray(solution.y_events[0][0], dtype=float)
    elif closest_states.size:
        closest_radii = np.linalg.norm(closest_states[:, :3], axis=1)
        closest_index = int(np.argmin(closest_radii))
        closest_state = closest_states[closest_index]
        if closest_radii[closest_index] <= R_switch_m * (1.0 + 1.0e-10):
            closest_time_s = float(solution.t_events[2][closest_index])
            rebuilt = _reconstruct_incoming_switch_state(
                closest_state[:3],
                closest_state[3:],
                m_dm=m_dm,
                eps=eps,
                R_switch_m=R_switch_m,
                invalid_z_floor_m=invalid_z_floor_m,
                settings=settings,
                environment=environment,
            )
            entered = bool(rebuilt["success"])
            reconstructed_message = str(rebuilt["message"])
            if entered:
                switch_time_s = closest_time_s + float(
                    rebuilt.get("switch_time_relative_s", np.nan)
                )
                status = "dm_only_entered_switch_via_closest_approach"
                event_state = np.concatenate(
                    (rebuilt["x_switch_m"], rebuilt["v_switch_m_s"])
                )
            else:
                status = "dm_only_closest_approach_rebuild_failure"
                event_state = np.full(6, np.nan)
        elif escaped:
            status = "dm_only_escaped_without_switch"
            event_state = np.full(6, np.nan)
        elif invalid:
            status = "dm_only_invalid_domain"
            event_state = np.full(6, np.nan)
        elif not solution.success:
            status = "dm_only_solver_failure"
            event_state = np.full(6, np.nan)
        else:
            status = "dm_only_timeout"
            event_state = np.full(6, np.nan)
    elif escaped:
        status = "dm_only_escaped_without_switch"
        event_state = np.full(6, np.nan)
    elif invalid:
        status = "dm_only_invalid_domain"
        event_state = np.full(6, np.nan)
    elif not solution.success:
        status = "dm_only_solver_failure"
        event_state = np.full(6, np.nan)
    else:
        status = "dm_only_timeout"
        event_state = np.full(6, np.nan)

    event_to_append = 0 if len(solution.t_events[0]) else 1 if escaped else 0
    times, states = _append_event_state(solution, event_to_append)
    final_state = states[:, -1]
    minimum_radius = _sampled_min_radius(states, slice(0, 3))
    if closest_states.size:
        minimum_radius = min(
            minimum_radius,
            float(np.min(np.linalg.norm(closest_states[:, :3], axis=1))),
        )

    message = str(solution.message)
    if reconstructed_message:
        message += "; switch reconstructed from closest approach: " + reconstructed_message

    result = {
        "dm_only_success": bool(solution.success) and not status.endswith("failure"),
        "dm_only_status": status,
        "dm_only_message": message,
        "entered_switch": bool(entered),
        "dm_only_runtime_s": time.perf_counter() - started,
        "dm_only_t_final_s": float(times[-1]),
        "dm_only_t_reach_R_switch_s": switch_time_s,
        "dm_only_t_arrival_estimate_s": arrival_estimate,
        "dm_only_t_max_s": t_max,
        "dm_only_initial_radius_m": r0,
        "dm_only_initial_radial_velocity_m_s": initial_radial_velocity,
        "dm_only_final_radius_m": float(np.linalg.norm(final_state[:3])),
        "dm_only_min_radius_m": minimum_radius,
        "dm_only_final_radial_velocity_m_s": radial_velocity(
            final_state[:3], final_state[3:]
        ),
        "x_dm_initial_m": x0.copy(),
        "v_dm_initial_m_s": v0.copy(),
        "x_dm_final_m": final_state[:3].copy(),
        "v_dm_final_m_s": final_state[3:].copy(),
        "x_switch_m": np.asarray(event_state[:3], dtype=float),
        "v_switch_m_s": np.asarray(event_state[3:], dtype=float),
    }
    if keep_trajectory:
        result["times_s"] = times
        result["states"] = states
    return result


# =============================================================================
# FULL ION-DM COUPLED PROPAGATION
# =============================================================================


def _ion_energy_modes(
    position_m: Array,
    velocity_m_s: Array,
    *,
    settings: FullSettings,
    environment: SimulationEnvironment,
) -> Array:
    position = np.asarray(position_m, dtype=float)
    velocity = np.asarray(velocity_m_s, dtype=float)
    kinetic = 0.5 * environment.m_ion * velocity**2
    if settings.use_harmonic_ion:
        omega = environment.omega_vec.reshape(3, 1)
        return kinetic + 0.5 * environment.m_ion * (omega * position) ** 2
    return kinetic


def run_full_ion_dm_after_switch(
    x_switch_m: Array,
    v_switch_m_s: Array,
    *,
    m_dm: float,
    eps: float,
    R_full_m: float,
    R_switch_m: float,
    settings: FullSettings,
    environment: SimulationEnvironment,
    invalid_z_floor_m: float | None,
    x0_ion_m: Array | None = None,
    v0_ion_m_s: Array | None = None,
    target_energy_J: float = 1.0e-27,
    keep_trajectory: bool = True,
) -> dict[str, Any]:
    """Run the full single-ion / single-DM coupled ODE after switch entry."""

    started = time.perf_counter()
    x_switch = _as_vector(x_switch_m, "x_switch_m")
    v_switch = _as_vector(v_switch_m_s, "v_switch_m_s")
    x_ion0 = np.zeros(3) if x0_ion_m is None else _as_vector(x0_ion_m, "x0_ion_m")
    v_ion0 = np.zeros(3) if v0_ion_m_s is None else _as_vector(v0_ion_m_s, "v0_ion_m_s")

    initial_state = np.concatenate((x_ion0, x_switch, v_ion0, v_switch))
    softening_sq = float(settings.coulomb_softening_m) ** 2

    def rhs(_time: float, state: Array) -> Array:
        x_ion = state[0:3]
        x_dm = state[3:6]
        v_ion = state[6:9]
        v_dm = state[9:12]

        if settings.use_harmonic_ion:
            a_ion = -environment.omega_vec**2 * x_ion
        else:
            ion_force = _trap_force_from_FDC_FRF(
                x_ion[None, :],
                mass=environment.m_ion,
                charge=environment.Z_ion,
                environment=environment,
            )[0]
            a_ion = ion_force / environment.m_ion

        dm_force = _trap_force_from_FDC_FRF(
            x_dm[None, :],
            mass=m_dm,
            charge=eps,
            environment=environment,
        )[0]
        a_dm = dm_force / float(m_dm)

        delta = x_ion - x_dm
        separation_sq = max(
            float(np.dot(delta, delta)) + softening_sq,
            np.finfo(float).tiny,
        )
        coulomb_force = (
            environment.K
            * environment.Z_ion
            * float(eps)
            * environment.e**2
            * delta
            * separation_sq ** (-1.5)
        )
        a_ion = a_ion + coulomb_force / environment.m_ion
        a_dm = a_dm - coulomb_force / float(m_dm)
        return np.concatenate((v_ion, v_dm, a_ion, a_dm))

    escape_radius = float(settings.escape_factor_from_R_switch) * float(R_switch_m)

    def escape_event(_time: float, state: Array) -> float:
        relative_position = state[3:6] - state[0:3]
        relative_velocity = state[9:12] - state[6:9]
        separation = float(np.linalg.norm(relative_position))
        if radial_velocity(relative_position, relative_velocity) <= 0.0:
            return -abs(separation - escape_radius) - 1.0e-30
        return separation - escape_radius

    escape_event.terminal = True
    escape_event.direction = 1

    def R_full_entry_event(_time: float, state: Array) -> float:
        return float(np.linalg.norm(state[3:6] - state[0:3]) - R_full_m)

    R_full_entry_event.terminal = False
    R_full_entry_event.direction = -1
    events: list[Callable] = [escape_event, R_full_entry_event]

    if invalid_z_floor_m is not None:
        def invalid_event(_time: float, state: Array) -> float:
            return float(min(state[2], state[5]) - invalid_z_floor_m)

        invalid_event.terminal = True
        invalid_event.direction = -1
        events.append(invalid_event)

    speed = max(float(np.linalg.norm(v_switch)), 1.0e-12)
    t_max = max(
        settings.time_factor * float(R_switch_m) / speed,
        settings.minimum_time_s,
    )
    t_eval = np.arange(0.0, t_max, settings.sample_dt_s, dtype=float)
    if t_eval.size == 0 or not np.isclose(t_eval[0], 0.0):
        t_eval = np.insert(t_eval, 0, 0.0)
    if not np.isclose(t_eval[-1], t_max):
        t_eval = np.append(t_eval, t_max)

    solution = solve_ivp(
        rhs,
        (0.0, t_max),
        initial_state,
        method="DOP853",
        t_eval=t_eval,
        rtol=settings.rtol,
        atol=settings.atol,
        max_step=settings.max_step_s,
        events=events,
    )

    escaped = len(solution.t_events[0]) > 0
    reached_R_full_event = len(solution.t_events[1]) > 0
    invalid_index = 2
    invalid = len(events) > invalid_index and len(solution.t_events[invalid_index]) > 0
    event_index = 0 if escaped else invalid_index if invalid else 0
    times, states = _append_event_state(solution, event_index)

    x_ion = states[0:3]
    x_dm = states[3:6]
    v_ion = states[6:9]
    v_dm = states[9:12]
    relative = x_dm - x_ion
    separations = np.linalg.norm(relative, axis=0)
    d_min = float(np.min(separations)) if separations.size else np.nan
    reached_R_full = bool(
        reached_R_full_event
        or (np.isfinite(d_min) and d_min <= float(R_full_m))
    )
    full_t_reach_R_full_s = _first_event_time(solution, 1)

    energy_modes = _ion_energy_modes(
        x_ion,
        v_ion,
        settings=settings,
        environment=environment,
    )
    total_energy = np.sum(energy_modes, axis=0)
    final_time = float(times[-1]) if times.size else 0.0
    average_mask = times >= max(
        0.0,
        final_time - settings.energy_average_window_s,
    )
    if not np.any(average_mask):
        average_mask = np.ones_like(times, dtype=bool)

    final_modes = np.mean(energy_modes[:, average_mask], axis=1)
    final_energy = float(np.sum(final_modes))
    initial_energy = float(total_energy[0]) if total_energy.size else 0.0
    delta_energy = final_energy - initial_energy

    interaction_switch = _time_inside(times, separations <= float(R_switch_m))
    interaction_full = _time_inside(times, separations <= float(R_full_m))

    if invalid:
        status = "full_invalid_domain"
    elif not solution.success:
        status = "full_solver_failure"
    elif not escaped:
        status = "full_timeout"
    elif reached_R_full:
        status = "reached_R_full_and_escaped"
    else:
        status = "entered_switch_but_missed_R_full"

    result = {
        "full_success": bool(solution.success),
        "full_status": status,
        "full_message": str(solution.message),
        "reached_R_full": reached_R_full,
        "above_energy_threshold": bool(
            status == "reached_R_full_and_escaped"
            and delta_energy >= float(target_energy_J)
        ),
        "d_min_m": d_min,
        "d_min_um": d_min * 1.0e6 if np.isfinite(d_min) else np.nan,
        "E_initial_J": initial_energy,
        "E_final_J": final_energy,
        "delta_E_ion_J": delta_energy,
        "E_x_J": float(final_modes[0]),
        "E_y_J": float(final_modes[1]),
        "E_z_J": float(final_modes[2]),
        "interaction_time_inside_R_switch_s": interaction_switch,
        "interaction_time_inside_R_full_s": interaction_full,
        "full_t_final_s": final_time,
        "full_t_reach_R_full_s": full_t_reach_R_full_s,
        "full_t_max_s": t_max,
        "full_runtime_s": time.perf_counter() - started,
        "x_ion_initial_m": x_ion0.copy(),
        "v_ion_initial_m_s": v_ion0.copy(),
        "x_dm_initial_m": x_switch.copy(),
        "v_dm_initial_m_s": v_switch.copy(),
        "x_ion_final_m": x_ion[:, -1].copy(),
        "v_ion_final_m_s": v_ion[:, -1].copy(),
        "x_dm_final_m": x_dm[:, -1].copy(),
        "v_dm_final_m_s": v_dm[:, -1].copy(),
    }
    if keep_trajectory:
        result.update(
            {
                "times_s": times,
                "states": states,
                "x_ion_m": x_ion,
                "v_ion_m_s": v_ion,
                "x_dm_m": x_dm,
                "v_dm_m_s": v_dm,
                "separation_m": separations,
                "ion_energy_modes_J": energy_modes,
                "ion_total_energy_J": total_energy,
            }
        )
    return result


# =============================================================================
# OPTIONAL RUTHERFORD R_FULL HELPER
# =============================================================================


def reduced_mass(m_dm: float, m_ion: float) -> float:
    if m_dm <= 0.0 or m_ion <= 0.0:
        raise ValueError("Masses must be positive")
    return float(m_dm * m_ion / (m_dm + m_ion))


def rutherford_r_min_threshold_m(
    *,
    speed_m_s: float,
    m_dm: float,
    eps: float,
    target_energy_J: float,
    environment: SimulationEnvironment,
) -> float:
    """Analytic Test 0/Test 1 Rutherford radius used as ``R_full``."""

    speed = float(speed_m_s)
    if speed <= 0.0 or target_energy_J <= 0.0:
        return float("nan")
    mu = reduced_mass(m_dm, environment.m_ion)
    minimum_speed = math.sqrt(
        target_energy_J * environment.m_ion / (2.0 * mu**2)
    )
    if speed < minimum_speed:
        return float("nan")

    coupling = abs(
        environment.K
        * environment.Z_ion
        * float(eps)
        * environment.e**2
    )
    if coupling <= 0.0:
        return float("nan")

    return (
        coupling / (mu * speed**2)
        + coupling / speed
        * math.sqrt(2.0 / (environment.m_ion * target_energy_J))
    )


# =============================================================================
# SUMMARY AND PLOTTING
# =============================================================================


def _print_stage_summary(result: dict[str, Any]) -> None:
    parameters = result["parameters"]
    straight = result["straight"]
    dm_only = result.get("dm_only")
    full = result.get("full")

    R_far = float(straight.get("R_far_actual_m", np.nan))
    R_switch = float(parameters.get("R_switch_m", np.nan))
    R_full = float(parameters.get("R_full_m", np.nan))
    t_far = float(parameters.get("t_reach_R_far_from_R_far_start_s", 0.0))
    t_switch = float(parameters.get("t_reach_R_switch_from_R_far_start_s", np.nan))
    t_full = float(parameters.get("t_reach_R_full_from_R_far_start_s", np.nan))

    print("\n-------- Staged Simulation Summary --------")
    print(
        "Radii:         "
        f"R_far={R_far * 1e6:.6f} um  "
        f"R_switch={R_switch * 1e6:.6f} um  "
        f"R_full={R_full * 1e6:.6f} um"
    )
    print(
        "Reach times:   "
        f"t(R_far)={t_far * 1e6:.6f} us  "
        f"t(R_switch)={t_switch * 1e6:.6f} us  "
        f"t(R_full)={t_full * 1e6:.6f} us  "
        "[time origin is the DM launch at R_far]"
    )
    outer_nominal = float(
        parameters.get("nominal_time_R_max_to_R_far_s", np.nan)
    )
    if np.isfinite(outer_nominal):
        print(
            "Outer-line diagnostic: nominal time from R_max to R_far="
            f"{outer_nominal * 1e6:.6f} us"
        )

    print(
        "Straight tail: "
        f"status={straight.get('outer_status')}  "
        f"v_far={straight.get('v_far_m_s', np.nan):.6g} m/s  "
        f"limiting={straight.get('limiting_criterion')}"
    )
    if dm_only is not None:
        print(
            "DM only:      "
            f"status={dm_only.get('dm_only_status')}  "
            f"entered_switch={dm_only.get('entered_switch')}  "
            f"r_min={dm_only.get('dm_only_min_radius_m', np.nan) * 1e6:.3f} um  "
            f"r_min/R_far={dm_only.get('dm_only_min_radius_over_R_far', np.nan):.6g}  "
            f"runtime={dm_only.get('dm_only_runtime_s', np.nan):.3f} s"
        )
    if full is not None:
        print(
            "Full coupled: "
            f"status={full.get('full_status')}  "
            f"reached_R_full={full.get('reached_R_full')}  "
            f"d_min={full.get('d_min_um', np.nan):.3f} um  "
            f"delta_E={full.get('delta_E_ion_J', np.nan):.6e} J  "
            f"runtime={full.get('full_runtime_s', np.nan):.3f} s"
        )

    print("\n-------- Initial and Final States --------")
    print("Positions are in um; velocities are in m/s.")
    print(
        "DM initial at R_far:  "
        f"x={_format_vector(parameters.get('x_dm_initial_at_R_far_m', np.full(3, np.nan)), scale=1e6)}  "
        f"v={_format_vector(parameters.get('v_dm_initial_at_R_far_m_s', np.full(3, np.nan)))}"
    )
    if dm_only is not None and dm_only.get("entered_switch", False):
        print(
            "DM at R_switch:     "
            f"x={_format_vector(dm_only.get('x_switch_m', np.full(3, np.nan)), scale=1e6)}  "
            f"v={_format_vector(dm_only.get('v_switch_m_s', np.full(3, np.nan)))}"
        )

    print(
        "Ion initial:          "
        f"x={_format_vector(parameters.get('x_ion_initial_m', np.zeros(3)), scale=1e6)}  "
        f"v={_format_vector(parameters.get('v_ion_initial_m_s', np.zeros(3)))}"
    )

    if full is not None:
        print(
            "Ion final:            "
            f"x={_format_vector(full.get('x_ion_final_m', np.full(3, np.nan)), scale=1e6)}  "
            f"v={_format_vector(full.get('v_ion_final_m_s', np.full(3, np.nan)))}"
        )
        print(
            "DM final:             "
            f"x={_format_vector(full.get('x_dm_final_m', np.full(3, np.nan)), scale=1e6)}  "
            f"v={_format_vector(full.get('v_dm_final_m_s', np.full(3, np.nan)))}"
        )
    elif dm_only is not None:
        print("Ion final:            not propagated because the full stage was not entered")
        print(
            "DM final:             "
            f"x={_format_vector(dm_only.get('x_dm_final_m', np.full(3, np.nan)), scale=1e6)}  "
            f"v={_format_vector(dm_only.get('v_dm_final_m_s', np.full(3, np.nan)))}"
        )
    else:
        print("Ion final:            not propagated")
        print("DM final:             not propagated beyond the straight-line diagnostic")

    print(f"\nOverall status: {result.get('overall_status')}")


def plot_simulation(result: dict[str, Any]) -> None:
    """Display diagnostic figures.  This function does not save files."""

    import matplotlib.pyplot as plt

    straight = result["straight"]
    xyz = np.asarray(straight["xyz_path_m"])
    radius = np.linalg.norm(xyz, axis=1)

    plt.figure(figsize=(8, 5))
    plt.semilogx(radius * 1.0e6, straight["energy_ratio_tail_max_array"], label="energy tail")
    plt.semilogx(radius * 1.0e6, straight["theta_abs_tail_array"], label="angle tail")
    plt.semilogx(
        radius * 1.0e6,
        np.asarray(straight["displacement_abs_tail_array_m"]) * 1.0e6,
        label="displacement tail [um]",
    )
    plt.axvline(straight["R_far_actual_m"] * 1.0e6, linestyle="--", label="R_far")
    plt.xlabel("Nominal path radius [um]")
    plt.ylabel("Diagnostic value")
    plt.title("Test 1 outer-tail diagnostics")
    plt.legend()
    plt.tight_layout()
    plt.show()

    dm_only = result.get("dm_only")
    if dm_only is not None and "times_s" in dm_only:
        states = np.asarray(dm_only["states"])
        dm_radius = np.linalg.norm(states[:3], axis=0)
        plt.figure(figsize=(8, 5))
        plt.plot(np.asarray(dm_only["times_s"]) * 1.0e6, dm_radius * 1.0e6)
        plt.axhline(result["parameters"]["R_switch_m"] * 1.0e6, linestyle="--", label="R_switch")
        plt.axhline(straight["R_far_actual_m"] * 1.0e6, linestyle=":", label="R_far")
        plt.xlabel("DM-only time [us]")
        plt.ylabel("DM radius [um]")
        plt.title("DM-only trap propagation")
        plt.legend()
        plt.tight_layout()
        plt.show()

    full = result.get("full")
    if full is not None and "times_s" in full:
        plt.figure(figsize=(8, 5))
        plt.plot(np.asarray(full["times_s"]) * 1.0e6, np.asarray(full["separation_m"]) * 1.0e6)
        plt.axhline(result["parameters"]["R_switch_m"] * 1.0e6, linestyle="--", label="R_switch")
        plt.axhline(result["parameters"]["R_full_m"] * 1.0e6, linestyle=":", label="R_full")
        plt.xlabel("Full-stage time [us]")
        plt.ylabel("Ion-DM separation [um]")
        plt.title("Full coupled ion-DM propagation")
        plt.legend()
        plt.tight_layout()
        plt.show()

        x_ion = np.asarray(full["x_ion_m"])
        energy = np.asarray(full["ion_total_energy_J"])
        times = np.asarray(full["times_s"])

        plt.figure(figsize=(8, 5))
        plt.plot(times * 1.0e6, energy)
        plt.xlabel("Full-stage time [us]")
        plt.ylabel("Ion energy [J]")
        plt.title("Ion energy versus time")
        plt.tight_layout()
        plt.show()

        x_ion_um = x_ion * 1.0e6
        plane_specs = (
            (0, 1, "x [um]", "y [um]", "Ion motion in the xy plane"),
            (1, 2, "y [um]", "z [um]", "Ion motion in the yz plane"),
            (0, 2, "x [um]", "z [um]", "Ion motion in the xz plane"),
        )
        for first, second, xlabel, ylabel, title in plane_specs:
            plt.figure(figsize=(7, 6))
            plt.plot(x_ion_um[first], x_ion_um[second], label="ion trajectory")
            plt.scatter(
                [x_ion_um[first, 0]],
                [x_ion_um[second, 0]],
                marker="o",
                label="initial ion",
            )
            plt.scatter(
                [x_ion_um[first, -1]],
                [x_ion_um[second, -1]],
                marker="x",
                label="final ion",
            )
            plt.xlabel(xlabel)
            plt.ylabel(ylabel)
            plt.title(title)
            plt.axis("equal")
            plt.legend()
            plt.tight_layout()
            plt.show()

        fig = plt.figure(figsize=(8, 6))
        axis = fig.add_subplot(projection="3d")
        x_dm = np.asarray(full["x_dm_m"]) * 1.0e6
        axis.plot(x_dm[0], x_dm[1], x_dm[2], label="DM")
        axis.plot(x_ion_um[0], x_ion_um[1], x_ion_um[2], label="ion")
        axis.set_xlabel("x [um]")
        axis.set_ylabel("y [um]")
        axis.set_zlabel("z [um]")
        axis.set_title("Full coupled trajectories")
        axis.legend()
        plt.tight_layout()
        plt.show()


# =============================================================================
# MAIN PUBLIC ENTRY POINT
# =============================================================================


def single_simulation(
    m_dm: float,
    eps: float,
    theta: float,
    alpha: float,
    psi: float,
    b: float,
    *,
    speed: float | None = None,
    T: float = 300.0,
    seed: int = 1,
    R_full: float | None = None,
    R_switch: float | None = None,
    R_switch_factor: float = 2.0,
    target_E: float = 1.0e-27,
    R_max: float = 20.0e-3,
    n_path: int = 800,
    energy_tol: float = 1.0e-2,
    angle_tol: float = 1.0e-3,
    displacement_tol: float = 1.0e-6,
    dm_only_rtol: float = 1.0e-6,
    dm_only_atol: float = 1.0e-9,
    dm_only_max_step: float = 5.0e-7,
    full_rtol: float = 1.0e-6,
    full_atol: float = 1.0e-9,
    full_dt: float = 5.0e-9,
    full_max_step: float = 5.0e-9,
    use_harmonic_ion: bool = True,
    coulomb_softening: float = 0.0,
    x0_ion: Array | None = None,
    v0_ion: Array | None = None,
    invalid_z_floor: float | None = None,
    z_margin: float = 1.0e-6,
    allow_R_far_lower_bound: bool = False,
    keep_trajectories: bool = True,
    print_summary: bool = True,
    plot: bool = False,
    environment: SimulationEnvironment | None = None,
) -> dict[str, Any]:
    """Run the complete straight-tail -> DM-only -> full coupled simulation.

    Parameters
    ----------
    m_dm, eps
        DM mass in kg and charge in units of the elementary charge.
    theta, alpha, psi, b
        Incoming direction and impact geometry.  Angles are radians and ``b``
        is in metres.
    speed
        Asymptotic speed ``v_inf`` in m/s.  If omitted, one Maxwell-Boltzmann
        3D velocity is sampled at temperature ``T`` and only its magnitude is
        used.
    R_full
        Full-interaction radius in metres.  If omitted, the analytic
        Rutherford threshold radius is calculated from ``speed`` and
        ``target_E``.
    R_switch
        DM-only/full-ODE handoff radius.  Defaults to
        ``R_switch_factor * R_full``.

    Returns
    -------
    dict
        Nested dictionaries named ``straight``, ``dm_only`` and ``full``.
        If the trajectory fails before a later stage, the corresponding later
        dictionary is ``None`` and ``overall_status`` explains why.
    """

    total_started = time.perf_counter()
    env = _resolve_environment(environment)

    m_dm = float(m_dm)
    eps = float(eps)
    b = float(b)
    if m_dm <= 0.0:
        raise ValueError("m_dm must be positive")
    if b < 0.0:
        raise ValueError("b must be nonnegative")

    if speed is None:
        if T <= 0.0:
            raise ValueError("T must be positive when speed is sampled")
        rng = np.random.default_rng(seed)
        sigma = math.sqrt(1.380649e-23 * float(T) / m_dm)
        velocity_sample = rng.normal(0.0, sigma, size=3)
        speed = float(np.linalg.norm(velocity_sample))
    else:
        speed = float(speed)
    if not np.isfinite(speed) or speed <= 0.0:
        raise ValueError("speed must be positive and finite")

    if R_full is None:
        R_full = rutherford_r_min_threshold_m(
            speed_m_s=speed,
            m_dm=m_dm,
            eps=eps,
            target_energy_J=target_E,
            environment=env,
        )
        if not np.isfinite(R_full):
            raise ValueError(
                "Could not derive R_full from the Rutherford threshold. "
                "Provide R_full explicitly or choose a speed above the "
                "kinematic threshold."
            )
    R_full = float(R_full)
    if R_full <= 0.0:
        raise ValueError("R_full must be positive")

    if R_switch is None:
        R_switch = float(R_switch_factor) * R_full
    R_switch = float(R_switch)
    if R_switch <= R_full:
        raise ValueError("R_switch must be greater than R_full")

    if invalid_z_floor is None and env.ion_height_m is not None:
        invalid_z_floor = -float(env.ion_height_m) + float(z_margin)

    outer_settings = OuterTailSettings(
        R_max_m=float(R_max),
        n_path=int(n_path),
        energy_tol=float(energy_tol),
        angle_tol=float(angle_tol),
        displacement_tol_m=float(displacement_tol),
        z_margin_m=float(z_margin),
        allow_R_far_lower_bound=bool(allow_R_far_lower_bound),
    )
    dm_settings = DMOnlySettings(
        rtol=float(dm_only_rtol),
        atol=float(dm_only_atol),
        max_step_s=float(dm_only_max_step),
    )
    full_settings = FullSettings(
        rtol=float(full_rtol),
        atol=float(full_atol),
        sample_dt_s=float(full_dt),
        max_step_s=float(full_max_step),
        use_harmonic_ion=bool(use_harmonic_ion),
        coulomb_softening_m=float(coulomb_softening),
    )

    parameters = {
        "m_dm_kg": m_dm,
        "eps": eps,
        "theta_rad": float(theta),
        "alpha_rad": float(alpha),
        "psi_rad": float(psi),
        "b_m": b,
        "b_um": b * 1.0e6,
        "v_inf_m_s": speed,
        "R_full_m": R_full,
        "R_full_um": R_full * 1.0e6,
        "R_switch_m": R_switch,
        "R_switch_um": R_switch * 1.0e6,
        "R_max_m": float(R_max),
        "target_E_J": float(target_E),
        "invalid_z_floor_m": invalid_z_floor,
        "x_ion_initial_m": (
            np.zeros(3, dtype=float)
            if x0_ion is None
            else _as_vector(x0_ion, "x0_ion").copy()
        ),
        "v_ion_initial_m_s": (
            np.zeros(3, dtype=float)
            if v0_ion is None
            else _as_vector(v0_ion, "v0_ion").copy()
        ),
        "t_reach_R_far_from_R_far_start_s": 0.0,
        "t_reach_R_switch_from_R_far_start_s": np.nan,
        "t_reach_R_full_from_R_far_start_s": np.nan,
    }

    straight = run_straight_outer_tail_test(
        theta=float(theta),
        alpha=float(alpha),
        psi=float(psi),
        b_m=b,
        v_inf_m_s=speed,
        m_dm=m_dm,
        eps=eps,
        R_full_m=R_full,
        settings=outer_settings,
        environment=env,
        invalid_z_floor_m=invalid_z_floor,
    )

    result: dict[str, Any] = {
        "parameters": parameters,
        "straight": straight,
        "dm_only": None,
        "full": None,
        "overall_status": "straight_stage_complete",
    }

    resolved_straight_statuses = {
        "straight_safe_to_full_sphere",
        "straight_then_dm_only",
    }
    if straight.get("outer_status") not in resolved_straight_statuses:
        if not (
            straight.get("outer_status") == "needs_larger_R_max"
            and allow_R_far_lower_bound
        ):
            result["overall_status"] = (
                "stopped_after_straight_stage:" + str(straight.get("outer_status"))
            )
            result["total_runtime_s"] = time.perf_counter() - total_started
            if print_summary:
                _print_stage_summary(result)
            if plot:
                plot_simulation(result)
            return result

    x_far = _as_vector(straight["xyz_far_m"], "straight xyz_far_m")
    v_far = _as_vector(
        straight["velocity_far_m_s"],
        "straight velocity_far_m_s",
    )
    parameters.update(
        {
            "R_far_path_coordinate_m": float(straight["R_far_m"]),
            "R_far_actual_m": float(straight["R_far_actual_m"]),
            "R_far_actual_um": float(straight["R_far_actual_m"]) * 1.0e6,
            "v_far_m_s": float(straight["v_far_m_s"]),
            "x_far_m": x_far.copy(),
            "v_far_vector_m_s": v_far.copy(),
            "initial_radial_velocity_at_R_far_m_s": radial_velocity(x_far, v_far),
            "x_dm_initial_at_R_far_m": x_far.copy(),
            "v_dm_initial_at_R_far_m_s": v_far.copy(),
            "nominal_time_R_max_to_R_far_s": max(
                float(np.max(np.asarray(straight.get("s_grid_m", [straight["R_far_m"]]))))
                - float(straight["R_far_m"]),
                0.0,
            ) / speed,
        }
    )

    dm_only = run_dm_trap_only_to_switch(
        x_far,
        v_far,
        m_dm=m_dm,
        eps=eps,
        R_switch_m=R_switch,
        settings=dm_settings,
        environment=env,
        invalid_z_floor_m=invalid_z_floor,
        keep_trajectory=keep_trajectories,
    )
    r_far_actual = float(straight["R_far_actual_m"])
    dm_only["dm_only_min_radius_over_R_far"] = (
        float(dm_only["dm_only_min_radius_m"]) / r_far_actual
        if r_far_actual > 0.0 and np.isfinite(dm_only["dm_only_min_radius_m"])
        else np.nan
    )
    dm_only["moved_inward_from_R_far"] = bool(
        np.isfinite(dm_only["dm_only_min_radius_m"])
        and dm_only["dm_only_min_radius_m"] < r_far_actual
    )
    result["dm_only"] = dm_only
    if dm_only.get("entered_switch", False):
        parameters["t_reach_R_switch_from_R_far_start_s"] = float(
            dm_only.get("dm_only_t_reach_R_switch_s", np.nan)
        )

    if not dm_only.get("entered_switch", False):
        result["overall_status"] = (
            "stopped_after_dm_only_stage:" + str(dm_only.get("dm_only_status"))
        )
        result["total_runtime_s"] = time.perf_counter() - total_started
        if print_summary:
            _print_stage_summary(result)
        if plot:
            plot_simulation(result)
        return result

    full = run_full_ion_dm_after_switch(
        dm_only["x_switch_m"],
        dm_only["v_switch_m_s"],
        m_dm=m_dm,
        eps=eps,
        R_full_m=R_full,
        R_switch_m=R_switch,
        settings=full_settings,
        environment=env,
        invalid_z_floor_m=invalid_z_floor,
        x0_ion_m=x0_ion,
        v0_ion_m_s=v0_ion,
        target_energy_J=target_E,
        keep_trajectory=keep_trajectories,
    )
    result["full"] = full
    t_switch_from_start = float(
        parameters.get("t_reach_R_switch_from_R_far_start_s", np.nan)
    )
    t_full_after_switch = float(full.get("full_t_reach_R_full_s", np.nan))
    if np.isfinite(t_switch_from_start) and np.isfinite(t_full_after_switch):
        parameters["t_reach_R_full_from_R_far_start_s"] = (
            t_switch_from_start + t_full_after_switch
        )
    result["overall_status"] = str(full.get("full_status"))
    result["total_runtime_s"] = time.perf_counter() - total_started

    if print_summary:
        _print_stage_summary(result)
    if plot:
        plot_simulation(result)
    return result


# =============================================================================
# EXAMPLE
# =============================================================================

# Example after running the trap-model notebook cells:
#
# result = single_simulation(
#     m_dm=2.636651e-25,
#     eps=2.335721,
#     theta=np.pi / 2,
#     alpha=0.0,
#     psi=0.0,
#     b=10.0e-6,
#     speed=1000.0,
#     R_full=61.340612e-6,
#     R_switch=122.681224e-6,
#     plot=True,
# )
#
# E_final = result["full"]["E_final_J"] if result["full"] is not None else np.nan